# Semana 8 — Cálculo Numérico
Resolução das quatro atividades do notebook do professor. Métodos usados: bisseção, Gauss-Seidel e Newton-Raphson.

Material de referência: [notebooks/semana8.ipynb do professor](https://github.com/walberbeltrame/cnum/blob/main/notebooks/semana8.ipynb).

## Atividade 1 — Temperatura mínima da placa
Resolver `5.67e-8*T**4 + 0.4*(T-K) - E = 0`, com `E = 500.125` e `K = 272.975`, pelo método da bisseção.

In [ ]:
E = 500.125
K = 272.975

def f(T):
    return 5.67e-8 * T**4 + 0.4 * (T - K) - E

def bissecao(f, a, b, tol=1e-12, max_iter=1000):
    if f(a) * f(b) > 0:
        raise ValueError('O intervalo precisa conter uma mudança de sinal.')
    for _ in range(max_iter):
        m = (a + b) / 2
        if abs(f(m)) < tol or (b - a) / 2 < tol:
            return m
        if f(a) * f(m) < 0:
            b = m
        else:
            a = m
    return (a + b) / 2

T = bissecao(f, 272.975, 400)
print(f'Temperatura mínima: {T:.12f} K')
print(f'Resíduo: {f(T):.3e}')

**Resultado esperado:** `T ≈ 304.568010119870 K`.

## Atividade 2 — Tensões dos três reatores
Aplicar Gauss-Seidel ao sistema linear com matriz diagonal dominante.

In [ ]:
import numpy as np

A = np.array([[17., -2., -3.],
              [-5., 21., -2.],
              [-5., -5., 22.]])
b = np.array([500., 200., 300.])

def gauss_seidel(A, b, tol=1e-10, max_iter=10000):
    x = np.zeros_like(b, dtype=float)
    for _ in range(max_iter):
        x_old = x.copy()
        for i in range(len(b)):
            soma1 = np.dot(A[i, :i], x[:i])
            soma2 = np.dot(A[i, i+1:], x_old[i+1:])
            x[i] = (b[i] - soma1 - soma2) / A[i, i]
        if np.linalg.norm(x - x_old, ord=np.inf) < tol:
            return x
    raise RuntimeError('O método não convergiu dentro do limite de iterações.')

R = gauss_seidel(A, b)
for i, valor in enumerate(R, start=1):
    print(f'R{i} = {valor:.12f}')
print('Resíduo:', np.linalg.norm(A @ R - b, ord=np.inf))

**Resultado esperado:** `R1 ≈ 36.560816557981`, `R2 ≈ 20.768358378225` e `R3 ≈ 26.665721576411`.

## Atividade 3 — Corrente no resistor R3
Resolver o sistema `[[20, 10], [10, 20]] [I1, I2] = [100, 100]` com Gauss-Seidel. Como os dois ramos contribuem para a corrente em `R3`, calcular `I_R3 = I1 + I2`.

In [ ]:
A3 = np.array([[20., 10.], [10., 20.]])
b3 = np.array([100., 100.])
I1, I2 = gauss_seidel(A3, b3)
I_R3 = I1 + I2
print(f'I1 = {I1:.6f} A')
print(f'I2 = {I2:.6f} A')
print(f'Corrente em R3 = {I_R3:.6f} A')

**Resultado esperado:** `I1 = I2 ≈ 3.333333 A`; portanto, `I_R3 ≈ 6.666667 A`.

## Atividade 4 — Sistema não linear de temperaturas
Usar Newton-Raphson para resolver `F(T1,T2) = 0`, com `E1 = 0.01753` e `E2 = 0.00254`. A cada iteração, resolver `J(T) Δ = -F(T)` e atualizar `T ← T + Δ`.

In [ ]:
E1, E2 = 0.01753, 0.00254

def F(T):
    t1, t2 = T
    f1 = t1**4 + 0.06823*t1 - t2**4 - 0.05848*t2 - E1
    f2 = t1**4 + 0.05848*t1 - 2*t2**4 - 0.11696*t2 - E2
    return np.array([f1, f2])

def jacobiana(T):
    t1, t2 = T
    return np.array([
        [4*t1**3 + 0.06823, -4*t2**3 - 0.05848],
        [4*t1**3 + 0.05848, -8*t2**3 - 0.11696]
    ])

def newton_raphson(T0, tol=1e-12, max_iter=100):
    T = np.array(T0, dtype=float)
    for k in range(max_iter):
        delta = np.linalg.solve(jacobiana(T), -F(T))
        T = T + delta
        if np.linalg.norm(delta, ord=np.inf) < tol:
            return T, k + 1
    raise RuntimeError('Newton-Raphson não convergiu.')

T, iteracoes = newton_raphson([0.3, 0.18])
print(f'T1 = {T[0]:.9f}')
print(f'T2 = {T[1]:.9f}')
print(f'Iterações: {iteracoes}')
print('Resíduo:', np.linalg.norm(F(T), ord=np.inf))

**Resultado esperado:** `T1 ≈ 0.30543` e `T2 ≈ 0.185261`.

## Conclusão
As quatro atividades foram implementadas com os métodos numéricos sugeridos. Execute as células em ordem para conferir os resultados e os resíduos numéricos.